In [ ]:
# ===== Colab环境配置 =====
# 运行此cell安装所有依赖（约2-3分钟）
!pip install -q numpy scipy matplotlib
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q soundfile librosa
!pip install -q deep-filter
!pip install -q pesq pystoi
print('环境配置完成！')

# 确认GPU
import torch
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('GPU不可用，请在运行时类型中选择T4 GPU')

# 下载DeepFilterNet代码
import os
if not os.path.exists('DeepFilterNet-main'):
    print('正在克隆DeepFilterNet仓库...')
    !git clone https://github.com/Rikorose/DeepFilterNet.git DeepFilterNet-main
    print('克隆完成！')

# 下载预训练模型
model_dir = 'DeepFilterNet-main/models'
if not os.path.exists(os.path.join(model_dir, 'DeepFilterNet3')):
    print('正在下载预训练模型...')
    !cd DeepFilterNet-main/models && unzip -q DeepFilterNet3.zip 2>/dev/null || echo '需要手动上传模型权重'

# 生成测试音频
print('环境准备就绪！')


# 模块5 第2次课：DeepFilterNet代码解析与运行本notebook包含：1. DeepFilterNet代码结构总览2. 配置参数分析3. 核心模块深度解析与数据流追踪4. 与DeepACE架构对比5. 使用预训练模型运行推理6. 输出分析与可视化---

## §1 代码结构总览DeepFilterNet-main 目录结构：```DeepFilterNet-main/+-- DeepFilterNet/df/        # Python包（核心代码）|   +-- deepfilternet.py     # DfNet模型 (Encoder, ErbDecoder, DfDecoder)|   +-- deepfilternet2.py    # DeepFilterNet2变体|   +-- deepfilternet3.py    # DeepFilterNet3变体|   +-- modules.py           # 基础组件 (Conv, GRU, Mask, DfOp)|   +-- config.py            # 配置系统 (DfParams)|   +-- enhance.py           # 推理入口|   +-- loss.py              # 损失函数 (MaskLoss, SpectralLoss等)|   +-- model.py             # 模型参数定义|   +-- train.py             # 训练脚本|   +-- io.py                # 音频I/O工具|   +-- utils.py             # 通用工具+-- models/                  # 预训练模型权重 (.zip)+-- assets/                  # 示例音频文件+-- libDF/                   # Rust核心 (STFT, ERB, 数据加载)+-- pyDF/                    # Rust核心的Python绑定```> "先见森林，再见树木"——先理解整体架构，再深入细节。

In [ ]:
# 加载配置并理解参数# DeepFilterNet 使用自定义配置系统# 这里展示关键的默认参数params = {    "SR": 48000,         # 采样率    "FFT_SIZE": 960,     # FFT窗口大小    "HOP_SIZE": 480,     # STFT步长 (50%重叠)    "NB_ERB": 32,        # ERB频带数    "NB_DF": 96,         # Deep Filtering频率bin数    "DF_ORDER": 5,       # Deep Filtering阶数（时域上下文）    "DF_LOOKAHEAD": 0,   # DF的前瞻帧数    "LSNR_MAX": 35,      # 最大局部SNR    "LSNR_MIN": -15,     # 最小局部SNR}print("=== DeepFilterNet 关键参数 ===")for k, v in params.items():    print("  %s: %s" % (k, v))# 推导值sr = params["SR"]fft_size = params["FFT_SIZE"]hop_size = params["HOP_SIZE"]freq_bins = fft_size // 2 + 1print()print("推导值:")print("  频率bin数: %d" % freq_bins)print("  帧时长: %.1f ms" % (fft_size / sr * 1000))print("  步长时长: %.1f ms" % (hop_size / sr * 1000))print("  每秒帧数: %.0f" % (sr / hop_size))

## §2 模型架构深度解析### DfNet 架构（来自 deepfilternet.py）主模型类 `DfNet` 有4个核心组件：1. **Encoder**：并行处理ERB特征和频谱特征   - ERB路径：4层Conv2d，通道数递增   - DF路径：2层Conv2d处理频谱特征   - 两条路径通过GroupedGRU融合，进行时序建模2. **ErbDecoder**：U-Net风格解码器，带跳跃连接   - 从编码特征重建ERB域掩码   - 输出：通过Sigmoid映射到 [0, 1] 的掩码3. **DfDecoder**：生成Deep Filtering系数   - 接收编码嵌入和频谱特征   - 输出：DF系数 (order x bins) + alpha（混合权重）4. **Mask + DfOp**：应用增强   - Mask通过ERB逆滤波器组扩展到全频分辨率   - DfOp对低频部分施加学习到的时域滤波

In [ ]:
# 数据流分析（张量形状追踪）# 基于 DeepFilterNet3 配置batch = 2T = 100       # 时间帧数F = freq_bins # 481Fe = params["NB_ERB"]  # 32 ERB频带Fc = params["NB_DF"]   # 96 DF频率binprint("=== DeepFilterNet 数据流 ===")print()print("输入特征:")print("  feat_erb:  (%d, 1, %d, %d)  [batch, 1, T, ERB频带]" % (batch, T, Fe))print("  feat_spec: (%d, 2, %d, %d)  [batch, 2(实/虚), T, DF_bins]" % (batch, T, Fc))print("  spec:      (%d, 1, %d, %d, 2) [batch, 1, T, 频率bin, 实/虚]" % (batch, T, F))print()# Encoderprint("Encoder:")print("  e0: (B, 16, T, Fe)     # ERB Conv0")print("  e1: (B, 32, T, Fe/2)   # ERB Conv1")print("  e2: (B, 64, T, Fe/4)   # ERB Conv2")print("  e3: (B, 64, T, Fe/4)   # ERB Conv3")print("  c0: (B, 16, T, Fc)     # DF Conv0")print("  c1: (B, 32, T, Fc)     # DF Conv1")print("  emb: (B, T, hidden)    # 通过GroupedGRU融合的嵌入")print("  lsnr: (B, T, 1)        # 局部SNR估计")print()# ErbDecoderprint("ErbDecoder (U-Net解码器):")print("  e3 + emb -> convt3 -> (B, 64, T, Fe/4)")print("  e2 + up   -> convt2 -> (B, 32, T, Fe/2)")print("  e1 + up   -> convt1 -> (B, 16, T, Fe)")print("  e0 + up   -> conv0  -> (B, 1, T, Fe)")print("  mask = Sigmoid(output)  # [0, 1] 掩码")print()# DfDecoderprint("DfDecoder:")print("  df_coefs: (B, T, O, F, 2)  # O=5 DF阶数, F=96 bins, 实/虚")print("  df_alpha: (B, T, 1)         # 混合权重")print()# 最终输出print("最终输出:")print("  enhanced_spec: (B, 1, T, F, 2)  # 增强后的复数语谱图")

## §3 关键模块分析### GroupedGRU- 将GRU分成G个独立组- 每组处理一部分特征- 在保持容量的同时减少参数- 类似模块2中的分组卷积思想### Mask 模块- 将ERB域掩码转换回全频率分辨率- 使用ERB逆滤波器组：mask_full = mask_erb @ erb_inv_fb- 掩码以乘法方式应用：enhanced = spec * mask### DfOp（Deep Filtering操作）- 对低频bin做学习到的时域FIR滤波- Y[f] = sum_{k=0}^{K-1} C[k,f] * X[f-k]  （复数乘法）- alpha控制混合：output = alpha * 滤波结果 + (1-alpha) * 掩码结果- 5种不同的实现变体，适配不同部署目标

## §4 与DeepACE对比| 方面 | DeepACE（模块4） | DeepFilterNet（模块5） ||------|-----------------|----------------------|| 任务 | 电极图预测 | 语音增强 || 输入 | 带噪语音波形 | 带噪语谱图 || 输出 | 22通道电极图 | 增强后语谱图 || 域 | 时域 (Conv1d) | 时频域 (Conv2d) || 编码器 | Conv1d（学习特征） | Conv2d（ERB特征） || 时序模型 | 膨胀卷积 + SE | GroupedGRU || 解码器 | ConvTranspose1d | U-Net + 跳跃连接 || 核心创新 | ChannelRebalancer | Deep Filtering阶段 || 采样率 | 16 kHz | 48 kHz || 实时性 | 是（因果） | 是（约5ms延迟） |> 两个模型共享"编码器-解码器-掩码"模式，但应用于不同域和不同目的。

## §5 运行推理DeepFilterNet 提供预训练模型，可直接处理带噪音频。运行推理的几种方式：1. 命令行：`python -m df.enhance <输入目录>`2. Python API：使用 `enhance()` 函数3. pip包：`deep-filter` 命令本notebook使用Python API方式。

In [ ]:
# 运行 DeepFilterNet 推理import osimport sysDFN_DIR = os.path.join('DeepFilterNet-main', 'DeepFilterNet')sys.path.insert(0, DFN_DIR)# 检查 DeepFilterNet 是否可导入try:    from df.config import config    config.use_defaults()    from df.model import ModelParams    p = ModelParams()    print("DeepFilterNet 配置加载成功！")    print("  SR: %d" % p.sr)    print("  FFT_SIZE: %d" % p.fft_size)    print("  HOP_SIZE: %d" % p.hop_size)    print("  NB_ERB: %d" % p.nb_erb)    print("  NB_DF: %d" % p.nb_df)    print("  DF_ORDER: %d" % p.df_order)    DF_AVAILABLE = Trueexcept ImportError as e:    print("DeepFilterNet 无法导入:", e)    print("安装方式: pip install deep-filter")    print("或: cd %s && pip install -e ." % DFN_DIR)    DF_AVAILABLE = False

In [ ]:
# 加载预训练模型并运行推理if DF_AVAILABLE:    try:        from df.enhance import init_df, enhance        from df.io import load_audio, save_audio        model, df_state, suffix, epoch = init_df(            os.path.join('DeepFilterNet-main', 'models', 'DeepFilterNet3'),            post_filter=True,            config_allow_defaults=True,        )        print("预训练模型加载成功！(epoch %d)" % epoch)        # 查找测试音频        test_dir = os.path.join('test_samples')        assets_dir = os.path.join('DeepFilterNet-main', 'assets')        test_files = []        if os.path.exists(test_dir):            for f in sorted(os.listdir(test_dir)):                if f.endswith('.wav'):                    test_files.append(os.path.join(test_dir, f))        if not test_files and os.path.exists(assets_dir):            noisy_file = os.path.join(assets_dir, 'noisy_snr0.wav')            if os.path.exists(noisy_file):                test_files.append(noisy_file)        if test_files:            print("找到 %d 个测试文件" % len(test_files))            for f in test_files[:3]:                print("  %s" % os.path.basename(f))        else:            print("未找到测试音频，请先运行 scripts/prepare_test_samples.py")        HAS_MODEL = True    except Exception as e:        print("模型加载失败:", e)        print("请确认已解压模型权重:")        print("  cd ../DeepFilterNet-main/models/ && unzip DeepFilterNet3.zip")        HAS_MODEL = Falseelse:    HAS_MODEL = False    print("跳过推理（DeepFilterNet不可用）")

In [ ]:
# 增强音频并可视化结果import numpy as npif HAS_MODEL and len(test_files) > 0:    # 处理第一个测试文件    audio, meta = load_audio(test_files[0], p.sr, 'cpu')    enhanced = enhance(model, df_state, audio, pad=True)    # 波形对比    fig, axes = plt.subplots(2, 1, figsize=(14, 5))    t_axis = np.arange(len(audio[0])) / p.sr    axes[0].plot(t_axis, audio[0].numpy())    axes[0].set_title("带噪输入: %s" % os.path.basename(test_files[0]))    axes[0].set_xlabel("时间 (s)")    t_axis_enh = np.arange(enhanced.shape[-1]) / p.sr    axes[1].plot(t_axis_enh, enhanced[0].cpu().numpy())    axes[1].set_title("增强输出")    axes[1].set_xlabel("时间 (s)")    plt.tight_layout()    plt.show()    # 语谱图对比    import torch    noisy_spec = torch.stft(audio[0], p.fft_size, p.hop_size, return_complex=True)    enh_spec = torch.stft(enhanced[0].cpu(), p.fft_size, p.hop_size, return_complex=True)    fig, axes = plt.subplots(1, 2, figsize=(14, 5))    axes[0].imshow(20*np.log10(noisy_spec.abs().numpy() + 1e-8),                   aspect='auto', origin='lower', cmap='magma')    axes[0].set_title("带噪语谱图")    axes[1].imshow(20*np.log10(enh_spec.abs().numpy() + 1e-8),                   aspect='auto', origin='lower', cmap='magma')    axes[1].set_title("增强后语谱图")    plt.tight_layout()    plt.show()else:    print("跳过增强可视化")    print("运行此演示的步骤:")    print("  1. 解压模型权重: models/DeepFilterNet3.zip")    print("  2. 生成测试样本: python scripts/prepare_test_samples.py")    print("  3. 重新运行此notebook")

## §6 ERB vs 梅尔滤波器组可视化理解DeepFilterNet为什么选择ERB而非梅尔：

In [ ]:
# 对比ERB和梅尔滤波器组import numpy as npimport matplotlib.pyplot as pltsr = 48000n_fft = 960freq_bins = n_fft // 2 + 1freqs = np.linspace(0, sr // 2, freq_bins)# 简化的梅尔滤波器组def mel_filterbank(n_mels, sr, n_fft):    n_freqs = n_fft // 2 + 1    freqs = np.linspace(0, sr // 2, n_freqs)    mel_freqs = 2595 * np.log10(1 + freqs / 700)    mel_points = np.linspace(mel_freqs[0], mel_freqs[-1], n_mels + 2)    hz_points = 700 * (10 ** (mel_points / 2595) - 1)    fb = np.zeros((n_mels, n_freqs))    for i in range(n_mels):        f_left = hz_points[i]        f_center = hz_points[i + 1]        f_right = hz_points[i + 2]        for j in range(n_freqs):            if freqs[j] >= f_left and freqs[j] <= f_center:                fb[i, j] = (freqs[j] - f_left) / (f_center - f_left + 1e-8)            elif freqs[j] > f_center and freqs[j] <= f_right:                fb[i, j] = (f_right - freqs[j]) / (f_right - f_center + 1e-8)    return fb# 简化的ERB滤波器组def erb_filterbank(n_erbs, sr, n_fft):    n_freqs = n_fft // 2 + 1    freqs = np.linspace(0, sr // 2, n_freqs)    erb_points = 21.4 * np.log10(0.00437 * freqs + 1)    erb_grid = np.linspace(erb_points[0], erb_points[-1], n_erbs + 2)    hz_grid = (10 ** (erb_grid / 21.4) - 1) / 0.00437    fb = np.zeros((n_erbs, n_freqs))    for i in range(n_erbs):        f_left = hz_grid[i]        f_center = hz_grid[i + 1]        f_right = hz_grid[i + 2]        for j in range(n_freqs):            if freqs[j] >= f_left and freqs[j] <= f_center:                fb[i, j] = (freqs[j] - f_left) / (f_center - f_left + 1e-8)            elif freqs[j] > f_center and freqs[j] <= f_right:                fb[i, j] = (f_right - freqs[j]) / (f_right - f_center + 1e-8)    return fbmel_fb = mel_filterbank(32, sr, n_fft)erb_fb = erb_filterbank(32, sr, n_fft)fig, axes = plt.subplots(1, 2, figsize=(14, 5))axes[0].imshow(mel_fb, aspect='auto', origin='lower', cmap='hot',               extent=[0, sr//2, 0, 32])axes[0].set_title("梅尔滤波器组 (32个频带)")axes[0].set_xlabel("频率 (Hz)")axes[0].set_ylabel("频带索引")axes[1].imshow(erb_fb, aspect='auto', origin='lower', cmap='hot',               extent=[0, sr//2, 0, 32])axes[1].set_title("ERB滤波器组 (32个频带)")axes[1].set_xlabel("频率 (Hz)")axes[1].set_ylabel("频带索引")plt.tight_layout()plt.show()print("观察：ERB滤波器组在低频分配更多频带")print("这与人耳耳蜗的频率分辨率模式一致")print("CI电极遵循类似的非线性频率映射")

---## 小结本节课我们：1. 了解了DeepFilterNet的代码结构和组织方式2. 理解了配置参数及其物理含义3. 追踪了模型架构中的详细数据流4. 对比了DeepFilterNet与DeepACE的架构异同5. 使用预训练模型运行了推理6. 可视化了ERB与梅尔滤波器组的差异**要点回顾**：- DeepFilterNet采用两阶段方法：基于掩码的增强 + 深度滤波- ERB域的选择源于其与人耳听觉感知的关联- 编码器-解码器模式与跳跃连接是DeepACE和DeepFilterNet共有的- 实时推理可行，延迟约5ms**课后任务**：- 阅读 `loss.py`，理解多组件损失函数- 尝试增强自己的音频文件- 预习下次课：思考如何将DeepFilterNet与ACE策略连接